# 00 · Setup Check — Verificação do Ambiente

🎯 **Objetivo:** Validar seu ambiente Python/PySpark.

Execute este notebook **antes de iniciar qualquer laboratório**. Ele é seguro para rodar a qualquer momento, com qualquer subconjunto de `make up-*` em execução (ou nenhum — o Caso A usa `local[*]` e não precisa de Docker).

💡 **Dica:** Use este notebook como um "check-up" sempre que trocar de máquina, recriar o ambiente virtual ou encontrar erros inesperados.

---
## ✅ O que este notebook verifica

1. **Versão do Python** — 3.12+ obrigatório
2. **Versão do PySpark** — 3.5.x obrigatório
3. **SparkSession local** — Caso A funcional
4. **Dataset Bronze** — dados gerados?

Vamos começar!


In [ ]:
import sys

# Exibe a versão do Python em execução no momento
print(f"Python: {sys.version}")
# Garante que a versão atende ao requisito mínimo (3.12+) — aborta se for inferior
assert sys.version_info >= (3, 12), "This lab targets Python 3.12+"
print("✅ Python version OK")

📌 **O que acabamos de verificar:** A versão do Python (3.12+) é importante porque o PySpark 3.5.x tira proveito de otimizações internas que só existem a partir desta versão. Se você estiver com uma versão anterior, atualize antes de prosseguir.

💡 **Dica:** Para verificar manualmente no terminal: `python --version`


In [ ]:
import pyspark

# Exibe a versão do PySpark instalada no ambiente
print(f"PySpark: {pyspark.__version__}")
# Verifica se é a versão 3.5.x — única série testada para estes laboratórios
assert pyspark.__version__.startswith("3.5"), "Expected PySpark 3.5.x"
print("✅ PySpark version OK")

📌 **PySpark 3.5.x** é a versão LTS atual do ecossistema Spark. Ela traz suporte a Spark Connect (usado nos Casos B/D), melhorias no Optimizer Catalyst e integração nativa com Pandas via `toPandas()`. Manter a versão correta evita incompatibilidades com as APIs utilizadas nos laboratórios.

# Caso A: modo local puro 

Nenhum Docker envolvido — o Spark vem embutido no pacote pip.O master("local[*]") roda Driver E Executors como threads neste mesmo processo — a topologia mais simples que o Spark suporta.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")     # Em local[*] isso controla quantas tarefas rodam em paralelo
    .getOrCreate()
)
# Exibe a representação da SparkSession criada
spark
print("✅ Local Spark session works (Case A ready)", spark)


✅ **Caso A pronto.** O modo `local[*]` cria uma SparkSession completa — com Driver e Executors rodando como **threads na sua máquina** — sem precisar de Docker. O `*` significa "use todos os núcleos disponíveis".

**Spark UI**: Acessar [http://localhost:4040/jobs/](http://localhost:4040/jobs/)


# Finalizando a conexão com o Apache Spark

Com a finalização da conexão o Apache Spark também será finalizado.

In [ ]:
spark.stop()

## Verificação do Dataset (Camada Bronze)

Os Casos B/C/D usam dados na camada Bronze (`data/bronze/`). A célula abaixo verifica se o dataset já foi gerado e, caso não exista, gera automaticamente uma amostra de escala `small`.

📌 A escala `small` é suficiente para todos os notebooks deste laboratório. Dataset maior só é necessário para testes de desempenho com dados volumosos.

In [ ]:
import subprocess
from pathlib import Path

# Caminho para a camada Bronze do Data Lake (dados gerados)
data_dir = Path("../data/bronze")
# Verifica se o diretório existe e não está vazio
if data_dir.exists() and any(data_dir.iterdir()):
    print(f"✅ Dataset found at {data_dir.resolve()}")
    # Confirma a presença de cada tabela esperada no dataset
    for table in ["empresas", "funcionarios", "vendas"]:
        print(f"   - {table}: {'present' if (data_dir / table).exists() else 'MISSING'}")
else:
    print("⬜ No dataset found. Generating scale=small now...")
    # Gera o dataset automaticamente com escala pequena via script dedicado
    subprocess.run(
        ["uv", "run", "python", "scripts/generate_dataset.py", "--scale", "small"],
        cwd="..",
        check=True,
    )

📌 **Estrutura do dataset (camada Bronze):**
- `empresas/` — cadastro de empresas com setor de atuação
- `funcionarios/` — funcionários com cargo, salário e data de admissão
- `vendas/` — transações com valor, data e referências aos funcionários

Os próximos notebooks lêem esses mesmos arquivos. Se o dataset foi gerado agora, pode prosseguir tranquilamente.

🏁 **Setup completo!** Você já pode abrir e executar o notebook 01.
